In [1]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.8 MB/s eta 0:00:00


In [2]:
import re
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import pandas as pd

df = pd.read_csv("quran_sahih.csv")
docs = df["text"].astype(str).tolist()
  # ===============================
# 1. Preprocessing
# ===============================

def clean_text(doc):
    doc = str(doc).lower()

    doc = re.sub(r"[^a-zA-Z\s]", " ", doc)
    tokens = doc.split()
    return " ".join(tokens)

docs_clean = [clean_text(doc) for doc in docs]

# ===============================
# 2. STOPWORDS — USE ONLY ENGLISH
# ===============================
stop_words = [
    # Standard English stopwords
    "a","about","above","after","again","against","all","am","an","and","any","are",
    "as","at","be","because","been","before","being","below","between","both","but",
    "by","can","could","did","do","does","doing","down","during","each","few","for",
    "from","had","has","have","having","he","her","here","hers","herself","him",
    "himself","his","how","i","if","in","into","is","it","its","itself","just","me",
    "more","most","my","myself","no","nor","not","of","off","on","once","only","or",
    "other","our","ours","ourselves","out","over","own","same","she","should","so",
    "some","such","than","that","the","their","theirs","them","themselves","then",
    "there","these","they","this","those","through","to","too","under","until","up",
    "very","was","we","were","what","when","where","which","while","who","whom","why",
    "will","with","would","you","your","yours","yourself","yourselves","upon","is","o",

    # Sahih-specific
    "indeed","then","thus","so","such","among","amongst","because","as for","so that",
    "those who","thereby","therein","thereafter","thereof","thereon","whereby",
    "wherein","whereof","whereupon","whatever","whichever","whomever","whenever",
    "wherever","anyone","anything","everyone","everything","either","neither",
    "whether",

    # Quranic formulas
    "say","o","s"

    # Structural Quran translation particles
    ,"hereafter","thereafter","except",
    "lest"
]

# ===============================
# 3. VECTORIZER
# ===============================
vectorizer = CountVectorizer(
    stop_words=stop_words,
    lowercase=True,
    ngram_range=(1,2),   # unigram only
    min_df=2,
    max_df=0.80
)

# ===============================
# 4. UMAP
# ===============================
umap_model = UMAP(
    n_neighbors=50,
    n_components=15,
    min_dist=0.3,
    metric='cosine',
    random_state=42
)

# ===============================
# 5. HDBSCAN
# ===============================
hdbscan_model = HDBSCAN(
    min_cluster_size=5,
    min_samples=2,
    cluster_selection_method='leaf',
    metric='euclidean'
)

# ===============================
# 6. EMBEDDINGS – YOU MUST USE THIS
# ===============================
embedding_model = SentenceTransformer(
    "paraphrase-multilingual-mpnet-base-v2"
)

# ===============================
# 7. BERTopic
# ===============================
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics="auto"
)

topics, probs = topic_model.fit_transform(docs_clean)
topic_model.visualize_topics()


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3472,-1_allah_people_us_said,"[allah, people, us, said, lord, one, come, day...",[because of that we decreed upon the children ...
1,0,828,0_allah_muhammad_earth_fear allah,"[allah, muhammad, earth, fear allah, wills, be...",[whatever is in the heavens and whatever is on...
2,1,123,1_merciful_might merciful_lord exalted_mercifu...,"[merciful, might merciful, lord exalted, merci...",[and indeed your lord he is the exalted in mig...
3,2,48,2_said lord_believing servants_day resurrected...,"[said lord, believing servants, day resurrecte...",[he said my lord make for me a sign he said yo...
4,3,40,3_pharaoh_pharaoh said_haman_go pharaoh,"[pharaoh, pharaoh said, haman, go pharaoh, hau...",[pharaoh said you believed moses before i gave...
...,...,...,...,...,...
169,168,5,168_mention later_favorable_favorable mention_...,"[mention later, favorable, favorable mention, ...",[and we left for him favorable mention among l...
170,169,5,169_name lord_exalt name_lord great_name,"[name lord, exalt name, lord great, name, exal...",[so exalt the name of your lord the most great...
171,170,5,170_given record_record_record right_hand,"[given record, record, record right, hand, rig...",[so as for he who is given his record in his r...
172,171,5,171_saving_feed_want_poor,"[saving, feed, want, poor, captive, spite love...",[and does not encourage the feeding of the poo...


In [4]:

new_topics = topic_model.reduce_outliers(documents = docs_clean, topics = topics, strategy="c-tf-idf")

# You MUST run this step to update the model's internal representations
topic_model.update_topics(docs_clean, topics = new_topics, vectorizer_model = vectorizer)

print("Outliers successfully reduced and reassigned using the c-TF-IDF strategy.")
print("\nUpdated Topic Information:")
# Check the count of the -1 topic to verify the reduction
(topic_model.get_topic_info())

2025-12-05 04:53:46,990 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers successfully reduced and reassigned using the c-TF-IDF strategy.

Updated Topic Information:


,Topic,Count,Name,Representation,Representative_Docs
0,-1,12,-1_coral_choose_diverse_alarmed,"[coral, choose, diverse, alarmed, distracted, ...",[because of that we decreed upon the children ...
1,0,1036,0_muhammad_messenger_wills_belongs,"[muhammad, messenger, wills, belongs, fear all...",[whatever is in the heavens and whatever is on...
2,1,160,1_merciful_merciful lord_lord exalted_might me...,"[merciful, merciful lord, lord exalted, might ...",[and indeed your lord he is the exalted in mig...
3,2,76,2_said lord_day resurrected_believing servants...,"[said lord, day resurrected, believing servant...",[he said my lord make for me a sign he said yo...
4,3,70,3_pharaoh_pharaoh said_people pharaoh_pharaoh ...,"[pharaoh, pharaoh said, people pharaoh, pharao...",[pharaoh said you believed moses before i gave...
...,...,...,...,...,...
169,168,9,168_later_generations_left_remaining,"[later, generations, left, remaining, mention,...",[and we left for him favorable mention among l...
170,169,23,169_exalt_name lord_name_exalt allah,"[exalt, name lord, name, exalt allah, lord gre...",[so exalt the name of your lord the most great...
171,170,22,170_record_hand_right hand_read,"[record, hand, right hand, read, right, drew, ...",[so as for he who is given his record in his r...
172,171,18,171_feed_want_feeding_poor,"[feed, want, feeding, poor, oaths, spite, feed...",[and does not encourage the feeding of the poo...


In [5]:
# Assuming you want to sub-cluster Topic 3
TARGET_TOPIC_ID = 0

# 1. Get the indices of all documents belonging to the target topic
target_indices = [i for i, topic_id in enumerate(topics) if topic_id == TARGET_TOPIC_ID]

# 2. Extract the documents for the sub-clustering
sub_cluster_docs = [docs_clean[i] for i in target_indices]

print(f"Number of documents in Topic {TARGET_TOPIC_ID}: {len(sub_cluster_docs)}")
# 1. Create a fresh BERTopic model instance
sub_topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer,
    # Re-use your UMAP/HDBSCAN models or create new ones for tuning
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics="auto"
)

# 2. Fit and Transform the subset of documents
sub_topics, sub_probs = sub_topic_model.fit_transform(sub_cluster_docs)

Number of documents in Topic 0: 828


In [6]:
sub_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,249,-1_said_people_good_sent,"[said, people, good, sent, believe, muhammad, ...",[it is that of which allah gives good tidings ...
1,0,102,0_punishment_fire_disbelievers_believe,"[punishment, fire, disbelievers, believe, neve...",[those who disbelieved and averted others from...
2,1,61,1_merciful_forgiving_forgiving merciful_allah ...,"[merciful, forgiving, forgiving merciful, alla...",[except for those who repent after that and co...
3,2,48,2_believed_prayer_let_fear allah,"[believed, prayer, let, fear allah, fear, witn...",[o you who have believed be persistently stand...
4,3,45,3_quran_revealed_verses_book,"[quran, revealed, verses, book, muhammad, reve...",[alif lam ra this is a book which we have reve...
5,4,28,4_worship_besides allah_besides_worship besides,"[worship, besides allah, besides, worship besi...",[you only worship besides allah idols and you ...
6,5,20,5_created_allah created_creation_sky,"[created, allah created, creation, sky, earth,...",[have you not seen that allah created the heav...
7,6,19,6_messenger_notification_obey_clear notification,"[messenger, notification, obey, clear notifica...",[and obey allah and obey the messenger but if ...
8,7,16,7_hypocrites_messenger_believers_allah messenger,"[hypocrites, messenger, believers, allah messe...",[and there are those hypocrites who took for t...
9,8,15,8_said_mary_wife_said lord,"[said, mary, wife, said lord, mention, sought,...",[mention o muhammad when the wife of imran sai...


In [7]:

new_sub_topics = sub_topic_model.reduce_outliers(documents = sub_cluster_docs, topics = sub_topics, strategy="c-tf-idf")

# You MUST run this step to update the model's internal representations
sub_topic_model.update_topics(sub_cluster_docs, topics = new_sub_topics, vectorizer_model = vectorizer)

print("Outliers successfully reduced and reassigned using the c-TF-IDF strategy.")
print("\nUpdated Topic Information:")
# Check the count of the -1 topic to verify the reduction
(sub_topic_model.get_topic_info())

2025-12-05 04:53:59,754 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers successfully reduced and reassigned using the c-TF-IDF strategy.

Updated Topic Information:


,Topic,Count,Name,Representation,Representative_Docs
0,0,119,0_punishment_disbelieved_never_disbelievers,"[punishment, disbelieved, never, disbelievers,...",[those who disbelieved and averted others from...
1,1,82,1_merciful_forgiving_forgiving merciful_allah ...,"[merciful, forgiving, forgiving merciful, alla...",[except for those who repent after that and co...
2,2,67,2_believed_prayer_let_fear,"[believed, prayer, let, fear, fear allah, may,...",[o you who have believed be persistently stand...
3,3,59,3_quran_muhammad_revealed_sent,"[quran, muhammad, revealed, sent, verses, book...",[alif lam ra this is a book which we have reve...
4,4,36,4_besides_worship_besides allah_commanded,"[besides, worship, besides allah, commanded, p...",[you only worship besides allah idols and you ...
5,5,24,5_created_allah created_creation_earth,"[created, allah created, creation, earth, sky,...",[have you not seen that allah created the heav...
6,6,29,6_messenger_turn away_away_notification,"[messenger, turn away, away, notification, tur...",[and obey allah and obey the messenger but if ...
7,7,18,7_hypocrites_messenger_surely_party,"[hypocrites, messenger, surely, party, turns, ...",[and there are those hypocrites who took for t...
8,8,29,8_said_mention_mary_wife,"[said, mention, mary, wife, said lord, allah s...",[mention o muhammad when the wife of imran sai...
9,9,17,9_fear allah_fear_allah obey_obey,"[fear allah, fear, allah obey, obey, treat, ra...","[so fear allah and obey me, so fear allah and ..."


In [9]:
SUB_TARGET_TOPIC_ID = 0  # pick a sub-topic from Level 1
sub_target_indices = [i for i, t in enumerate(sub_topics) if t == SUB_TARGET_TOPIC_ID]
sub_sub_cluster_docs = [sub_cluster_docs[i] for i in sub_target_indices]

print(f"Level 2: Number of docs in Sub-Topic {SUB_TARGET_TOPIC_ID} = {len(sub_sub_cluster_docs)}")

# --- Level 2: sub-sub-cluster ---
sub_sub_topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics="auto"
)

sub_sub_topics, sub_sub_probs = sub_sub_topic_model.fit_transform(sub_sub_cluster_docs)
sub_sub_topic_model.get_topic_info()

Level 2: Number of docs in Sub-Topic 0 = 102


,Topic,Count,Name,Representation,Representative_Docs
0,-1,28,-1_lord_punishment_said_taken,"[lord, punishment, said, taken, disbelievers, ...",[so let not their wealth or their children imp...
1,0,16,0_people_revealed_way allah_way,"[people, revealed, way allah, way, sins, avert...",[they took their false oaths as a cover so the...
2,1,13,1_punishment_cursed_allah messenger_messenger,"[punishment, cursed, allah messenger, messenge...",[allah has promised the hypocrite men and hypo...
3,2,11,2_believe_believe allah_came_assumption,"[believe, believe allah, came, assumption, ass...",[and they swear by allah their strongest oaths...
4,3,8,3_land_secure_earth_come,"[land, secure, earth, come, allah one, power, ...",[have they not traveled through the land and s...
5,4,7,4_fire_disbelieve_never_appease,"[fire, disbelieve, never, appease, fire abide,...",[it is not for the polytheists to maintain the...
6,5,7,5_let_kill_punish_world,"[let, kill, punish, world, punishment, behind,...",[and let those executors and guardians fear in...
7,6,7,6_allah allah_sight_behind_equal,"[allah allah, sight, behind, equal, equal alla...",[descendants some of them from others and alla...
8,7,5,7_never_faith_disbelief_whoever,"[never, faith, disbelief, whoever, never harm,...",[indeed those who disbelieve and commit wrong ...


In [11]:

new_sub_sub_topics = sub_sub_topic_model.reduce_outliers(documents = sub_sub_cluster_docs, topics = sub_sub_topics, strategy="c-tf-idf")

# You MUST run this step to update the model's internal representations
sub_sub_topic_model.update_topics(sub_sub_cluster_docs, topics = new_sub_sub_topics, vectorizer_model = vectorizer)

print("Outliers successfully reduced and reassigned using the c-TF-IDF strategy.")
print("\nUpdated Topic Information:")
# Check the count of the -1 topic to verify the reduction
(sub_sub_topic_model.get_topic_info())

2025-12-05 04:55:23,368 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers successfully reduced and reassigned using the c-TF-IDF strategy.

Updated Topic Information:


,Topic,Count,Name,Representation,Representative_Docs
0,0,22,0_people_way allah_taken_way,"[people, way allah, taken, way, deeds, wrath, ...",[they took their false oaths as a cover so the...
1,1,17,1_punishment_hell_whoever_messenger,"[punishment, hell, whoever, messenger, abide, ...",[allah has promised the hypocrite men and hypo...
2,2,16,2_believe_came_knowing_day,"[believe, came, knowing, day, lord, said, assu...",[and they swear by allah their strongest oaths...
3,3,11,3_land_come_said_secure,"[land, come, said, secure, earth, even, allah ...",[have they not traveled through the land and s...
4,4,9,4_fire_disbelieve_never_fire abide,"[fire, disbelieve, never, fire abide, wealth, ...",[it is not for the polytheists to maintain the...
5,5,10,5_let_punish_disbelievers_allah intends,"[let, punish, disbelievers, allah intends, int...",[and let those executors and guardians fear in...
6,6,8,6_another_allah another_another deity_deity,"[another, allah another, another deity, deity,...",[descendants some of them from others and alla...
7,7,9,7_never_painful_disbelief_painful punishment,"[never, painful, disbelief, painful punishment...",[indeed those who disbelieve and commit wrong ...
